# Count

In [ ]:
from pathlib import Path
import pandas as pd
import time
import gc

# ============================================================
# IPEDS UNIQUE DEGREE / FIELD / GROUP COUNT
# PRINT TABLE ONLY
# NO SAVE
# MEMORY OPTIMIZED
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

CHUNK_SIZE = 200_000

print("=" * 100)
print("IPEDS UNIQUE DEGREE / FIELD / GROUP COUNT")
print("PRINT ONLY - NO SAVE")
print("=" * 100)
print("Source file:", DEGREE_FILE)

start_time = time.time()

# ------------------------------------------------------------
# Display setting
# ------------------------------------------------------------
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ------------------------------------------------------------
# Read columns only
# ------------------------------------------------------------
all_columns = pd.read_csv(DEGREE_FILE, nrows=0, low_memory=False).columns.tolist()

wanted_cols = [
    "year",
    "AWLEVEL",
    "CTOTALT",
    "cipcode",
    "CIPCODE",
    "cip_code",
    "major_name"
]

usecols = [c for c in wanted_cols if c in all_columns]

print("\nReading only columns:")
for c in usecols:
    print("-", c)

# ------------------------------------------------------------
# Degree level names
# ------------------------------------------------------------
AWLEVEL_MAP = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate degree",
    4: "Certificate 2-4 years",
    5: "Bachelor degree",
    6: "Post-bachelor certificate",
    7: "Master degree",
    8: "Post-master certificate",
    9: "Doctor degree",
    17: "Doctor degree - professional practice",
    18: "Doctor degree - research/scholarship",
    19: "Doctor degree - other",
    20: "Post-baccalaureate certificate",
    21: "Post-master certificate"
}

# ------------------------------------------------------------
# CIP group names
# 2-digit group
# ------------------------------------------------------------
CIP_GROUP_MAP = {
    "01": "Agriculture",
    "03": "Natural Resources",
    "04": "Architecture",
    "05": "Area, Ethnic, Cultural, Gender Studies",
    "09": "Communication and Journalism",
    "10": "Communications Technologies",
    "11": "Computer and Information Sciences",
    "12": "Personal and Culinary Services",
    "13": "Education",
    "14": "Engineering",
    "15": "Engineering Technologies",
    "16": "Foreign Languages",
    "19": "Family and Consumer Sciences",
    "22": "Legal Professions",
    "23": "English Language and Literature",
    "24": "Liberal Arts and Sciences",
    "25": "Library Science",
    "26": "Biological and Biomedical Sciences",
    "27": "Mathematics and Statistics",
    "28": "Military Science",
    "29": "Military Technologies",
    "30": "Multi/Interdisciplinary Studies",
    "31": "Parks, Recreation, Leisure, Fitness",
    "32": "Basic Skills",
    "33": "Citizenship Activities",
    "34": "Health-Related Knowledge and Skills",
    "35": "Interpersonal and Social Skills",
    "36": "Leisure and Recreational Activities",
    "37": "Personal Awareness",
    "38": "Philosophy and Religious Studies",
    "39": "Theology and Religious Vocations",
    "40": "Physical Sciences",
    "41": "Science Technologies",
    "42": "Psychology",
    "43": "Homeland Security, Law Enforcement, Firefighting",
    "44": "Public Administration and Social Service",
    "45": "Social Sciences",
    "46": "Construction Trades",
    "47": "Mechanic and Repair Technologies",
    "48": "Precision Production",
    "49": "Transportation and Materials Moving",
    "50": "Visual and Performing Arts",
    "51": "Health Professions",
    "52": "Business, Management, Marketing",
    "54": "History",
    "60": "Residency Programs"
}

# ------------------------------------------------------------
# Store summary only
# ------------------------------------------------------------
degree_parts = []
group_parts = []
field_parts = []
major_parts = []
degree_group_parts = []

chunk_num = 0
total_rows_seen = 0

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
):
    chunk_num += 1
    total_rows_seen += len(chunk)

    # -----------------------------
    # Clean count
    # -----------------------------
    chunk["CTOTALT"] = (
        chunk["CTOTALT"]
        .fillna("0")
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("*", "", regex=False)
        .str.strip()
    )
    chunk["CTOTALT"] = pd.to_numeric(chunk["CTOTALT"], errors="coerce").fillna(0)

    # -----------------------------
    # Clean AWLEVEL
    # Makes 5 and 5.0 the same
    # -----------------------------
    chunk["AWLEVEL_CLEAN"] = pd.to_numeric(chunk["AWLEVEL"], errors="coerce").astype("Int64")
    chunk["degree_level_name"] = chunk["AWLEVEL_CLEAN"].map(AWLEVEL_MAP).fillna("Unknown / blank")

    # -----------------------------
    # Clean CIP code
    # -----------------------------
    cip_source_cols = [c for c in ["cip_code", "CIPCODE", "cipcode"] if c in chunk.columns]

    chunk["cip_final"] = ""

    for c in cip_source_cols:
        temp = (
            chunk[c]
            .fillna("")
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.replace("-", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.strip()
        )

        chunk["cip_final"] = chunk["cip_final"].mask(chunk["cip_final"].eq(""), temp)

    chunk["cip_final"] = chunk["cip_final"].str.extract(r"(\d+)", expand=False).fillna("")

    chunk["group_code_2_digit"] = chunk["cip_final"].str[:2]
    chunk["field_code_4_digit"] = chunk["cip_final"].str[:4]
    chunk["major_code_6_digit"] = chunk["cip_final"].str[:6]

    chunk["field_group_name"] = chunk["group_code_2_digit"].map(CIP_GROUP_MAP).fillna("Unknown / blank")

    if "major_name" not in chunk.columns:
        chunk["major_name"] = "Unknown / blank"

    chunk["major_name"] = chunk["major_name"].fillna("Unknown / blank").astype(str).str.strip()

    # ========================================================
    # Summary 1: Degree level count
    # ========================================================
    degree_summary = (
        chunk
        .groupby(["AWLEVEL_CLEAN", "degree_level_name"], dropna=False)
        .agg(
            row_count=("CTOTALT", "size"),
            total_count=("CTOTALT", "sum")
        )
        .reset_index()
    )
    degree_parts.append(degree_summary)

    # ========================================================
    # Summary 2: Field group count
    # 2-digit CIP
    # Example: 50 = Visual and Performing Arts
    # ========================================================
    group_summary = (
        chunk
        .groupby(["group_code_2_digit", "field_group_name"], dropna=False)
        .agg(
            row_count=("CTOTALT", "size"),
            total_count=("CTOTALT", "sum")
        )
        .reset_index()
    )
    group_parts.append(group_summary)

    # ========================================================
    # Summary 3: Field count
    # 4-digit CIP
    # Example: 5003 = Dance
    # ========================================================
    field_summary = (
        chunk
        .groupby(["field_code_4_digit"], dropna=False)
        .agg(
            row_count=("CTOTALT", "size"),
            total_count=("CTOTALT", "sum")
        )
        .reset_index()
    )
    field_parts.append(field_summary)

    # ========================================================
    # Summary 4: Unique major count
    # 6-digit CIP + major name
    # Example: 500301 = Dance
    # ========================================================
    major_summary = (
        chunk
        .groupby(["major_code_6_digit", "major_name"], dropna=False)
        .agg(
            row_count=("CTOTALT", "size"),
            total_count=("CTOTALT", "sum")
        )
        .reset_index()
    )
    major_parts.append(major_summary)

    # ========================================================
    # Summary 5: Degree by field group count
    # ========================================================
    degree_group_summary = (
        chunk
        .groupby(
            ["AWLEVEL_CLEAN", "degree_level_name", "group_code_2_digit", "field_group_name"],
            dropna=False
        )
        .agg(
            row_count=("CTOTALT", "size"),
            total_count=("CTOTALT", "sum")
        )
        .reset_index()
    )
    degree_group_parts.append(degree_group_summary)

    if chunk_num == 1 or chunk_num % 5 == 0:
        elapsed = (time.time() - start_time) / 60
        print(
            f"Chunk {chunk_num} done | rows seen: {total_rows_seen:,} | total min: {elapsed:.1f}",
            flush=True
        )

    del chunk
    gc.collect()

# ============================================================
# Combine summary tables
# ============================================================

print("\n" + "=" * 100)
print("COMBINING TABLES")
print("=" * 100)

degree_count = (
    pd.concat(degree_parts, ignore_index=True)
    .groupby(["AWLEVEL_CLEAN", "degree_level_name"], dropna=False)
    .agg(
        row_count=("row_count", "sum"),
        total_count=("total_count", "sum")
    )
    .reset_index()
    .sort_values("total_count", ascending=False)
)

field_group_count = (
    pd.concat(group_parts, ignore_index=True)
    .groupby(["group_code_2_digit", "field_group_name"], dropna=False)
    .agg(
        row_count=("row_count", "sum"),
        total_count=("total_count", "sum")
    )
    .reset_index()
    .sort_values("total_count", ascending=False)
)

field_count = (
    pd.concat(field_parts, ignore_index=True)
    .groupby(["field_code_4_digit"], dropna=False)
    .agg(
        row_count=("row_count", "sum"),
        total_count=("total_count", "sum")
    )
    .reset_index()
    .sort_values("total_count", ascending=False)
)

unique_major_count = (
    pd.concat(major_parts, ignore_index=True)
    .groupby(["major_code_6_digit", "major_name"], dropna=False)
    .agg(
        row_count=("row_count", "sum"),
        total_count=("total_count", "sum")
    )
    .reset_index()
    .sort_values("total_count", ascending=False)
)

degree_by_field_group_count = (
    pd.concat(degree_group_parts, ignore_index=True)
    .groupby(
        ["AWLEVEL_CLEAN", "degree_level_name", "group_code_2_digit", "field_group_name"],
        dropna=False
    )
    .agg(
        row_count=("row_count", "sum"),
        total_count=("total_count", "sum")
    )
    .reset_index()
    .sort_values("total_count", ascending=False)
)

# ============================================================
# Print tables only
# ============================================================

print("\n" + "=" * 100)
print("TABLE 1: UNIQUE DEGREE LEVEL COUNT")
print("=" * 100)
display(degree_count)

print("\n" + "=" * 100)
print("TABLE 2: UNIQUE FIELD GROUP COUNT")
print("2-digit CIP group")
print("=" * 100)
display(field_group_count)

print("\n" + "=" * 100)
print("TABLE 3: UNIQUE FIELD COUNT")
print("4-digit CIP field")
print("=" * 100)
display(field_count)

print("\n" + "=" * 100)
print("TABLE 4: UNIQUE MAJOR COUNT")
print("6-digit CIP major")
print("=" * 100)
display(unique_major_count)

print("\n" + "=" * 100)
print("TABLE 5: DEGREE LEVEL BY FIELD GROUP COUNT")
print("=" * 100)
display(degree_by_field_group_count)

elapsed = (time.time() - start_time) / 60

print("\nDONE")
print(f"Total rows scanned: {total_rows_seen:,}")
print(f"Total time: {elapsed:.2f} minutes")
print("No file saved.")


# Count 2

In [ ]:
from pathlib import Path
import pandas as pd
import time
import gc

# ============================================================
# QUICK UNIQUE SUMMARY ONLY
# NO SAVE
# MEMORY OPTIMIZED
# ============================================================

DEGREE_FILE = Path.home() / "Downloads/Internship_SCIPE CI-SIP/MainProject/FullTest/All_Data/IPEDS_ALL_ORIGINAL_INFO_FIXED_V6.csv"

CHUNK_SIZE = 200_000

print("=" * 100)
print("QUICK UNIQUE SUMMARY")
print("=" * 100)

start_time = time.time()

all_columns = pd.read_csv(DEGREE_FILE, nrows=0, low_memory=False).columns.tolist()

usecols = [
    c for c in [
        "year",
        "AWLEVEL",
        "CTOTALT",
        "cipcode",
        "CIPCODE",
        "cip_code",
        "major_name"
    ]
    if c in all_columns
]

unique_years = set()
unique_degree_levels = set()
unique_field_groups_2_digit = set()
unique_fields_4_digit = set()
unique_majors_6_digit = set()
unique_major_names = set()

total_rows = 0
total_degree_count = 0

chunk_num = 0

for chunk in pd.read_csv(
    DEGREE_FILE,
    usecols=usecols,
    chunksize=CHUNK_SIZE,
    dtype=str,
    low_memory=False
):
    chunk_num += 1
    total_rows += len(chunk)

    # Year
    if "year" in chunk.columns:
        years = pd.to_numeric(chunk["year"], errors="coerce").dropna().astype(int)
        unique_years.update(years.unique())

    # Degree level
    if "AWLEVEL" in chunk.columns:
        aw = pd.to_numeric(chunk["AWLEVEL"], errors="coerce").dropna().astype(int)
        unique_degree_levels.update(aw.unique())

    # Total degree count
    if "CTOTALT" in chunk.columns:
        count = (
            chunk["CTOTALT"]
            .fillna("0")
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("*", "", regex=False)
            .str.strip()
        )
        count = pd.to_numeric(count, errors="coerce").fillna(0)
        total_degree_count += count.sum()

    # Clean CIP code
    cip_source_cols = [c for c in ["cip_code", "CIPCODE", "cipcode"] if c in chunk.columns]

    cip_final = pd.Series("", index=chunk.index, dtype="string")

    for c in cip_source_cols:
        temp = (
            chunk[c]
            .fillna("")
            .astype(str)
            .str.replace(".", "", regex=False)
            .str.replace("-", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.strip()
        )

        cip_final = cip_final.mask(cip_final.eq(""), temp)

    cip_final = cip_final.str.extract(r"(\d+)", expand=False).fillna("")

    valid_cip = cip_final[cip_final != ""]

    unique_field_groups_2_digit.update(valid_cip.str[:2].unique())
    unique_fields_4_digit.update(valid_cip.str[:4].unique())
    unique_majors_6_digit.update(valid_cip.str[:6].unique())

    # Major name
    if "major_name" in chunk.columns:
        names = chunk["major_name"].fillna("").astype(str).str.strip()
        names = names[names != ""]
        unique_major_names.update(names.unique())

    if chunk_num == 1 or chunk_num % 5 == 0:
        print(f"Chunk {chunk_num} done | rows scanned: {total_rows:,}", flush=True)

    del chunk
    gc.collect()

print("\n" + "=" * 100)
print("QUICK SUMMARY RESULT")
print("=" * 100)

print(f"Total rows scanned: {total_rows:,}")
print(f"Total degree/completion count CTOTALT: {total_degree_count:,.0f}")

print("\nUnique counts:")
print(f"Unique years: {len(unique_years):,}")
print(f"Unique degree levels AWLEVEL: {len(unique_degree_levels):,}")
print(f"Unique field groups, 2-digit CIP: {len(unique_field_groups_2_digit):,}")
print(f"Unique fields, 4-digit CIP: {len(unique_fields_4_digit):,}")
print(f"Unique majors, 6-digit CIP: {len(unique_majors_6_digit):,}")
print(f"Unique major names: {len(unique_major_names):,}")

print("\nYear range:")
print(f"{min(unique_years)} - {max(unique_years)}")

print("\nDegree levels found:")
print(sorted(unique_degree_levels))

print("\nField group codes found:")
print(sorted(unique_field_groups_2_digit))

elapsed = (time.time() - start_time) / 60
print(f"\nDONE | Total time: {elapsed:.2f} minutes")
print("No file saved.")